# Két lemez közös éllel

A két síklemez közös Gmsh-élhez csatlakozik. A középfelület elmozdulása és a fizikai forgás két háromkomponensű globális mező. A helyi alakváltozásokat lemezenként vetítjük ki.

In [1]:
using LowLevelFEM, LinearAlgebra, SparseArrays

geo = joinpath(pkgdir(LowLevelFEM), "examples", "dev", "shells", "folded-plate.geo")
isfile(geo) || error("Missing geometry: $geo")
openGeometry(geo)

In [2]:
surfaces = ["horizontal", "vertical"]
mat = [Material(surface; E=2e5, ν=0.3) for surface in surfaces]

U = Field(mat; type=:VectorField, dim=3, field=:u)
Φ = Field(mat; type=:VectorField, dim=3, field=:φ)

Problem("folded-plate", :VectorField, 3, 3, Material[Material("horizontal", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0), Material("vertical", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 325, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :φ, :rhs, false, :lagrange, LowLevelFEM.ApproximationData(nothing, nothing, nothing))

Egy síklemezen az irányok állandók, ezért mindegyik mezőből elég az első elemi csomópont értéke.

In [3]:
normals = [normalVector(U, surface) for surface in surfaces]
tangents = [tangentVectors(U, surface) for surface in surfaces]

n = [v.A[1][1:3] for v in normals]
t1 = [v1.A[1][1:3] for (v1, v2) in tangents]
t2 = [v2.A[1][1:3] for (v1, v2) in tangents]

2-element Vector{Vector{Float64}}:
 [0.0, 1.0, 0.0]
 [0.0, 0.0, -1.0]

In [4]:
b = 0.12
k_s = 4/5

K_plate = SystemMatrix[]
for (i, surface) in pairs(surfaces)
    tξ, tη, normal = t1[i], t2[i], n[i]

    # kron(tξ, tη)' ⋅ Grad(U) gives tξᵀ(∇U)tη.
    A_u = vcat(kron(tξ,tξ)',
               kron(tη,tη)',
               (kron(tξ,tη) + kron(tη,tξ))',
               zeros(3,9),
               kron(normal,tη)',
               kron(normal,tξ)')

    A_φ = vcat(zeros(3,9),
               kron(tη,tξ)',
               -kron(tξ,tη)',
               (kron(tη,tη) - kron(tξ,tξ))',
               zeros(2,9))
    G_φ = vcat(zeros(6,3), -tξ', tη')

    B = A_u ⋅ Grad(U) + A_φ ⋅ Grad(Φ) + G_φ ⋅ Φ

    C = D(:PlaneStress, mat[i])
    D1 = b*C
    D2 = b^3/12*C
    D3 = k_s*b*mat[i].μ * Matrix{Float64}(I, 2, 2)
    D_plate = cat(D1, D2, D3; dims=(1,2))
    push!(K_plate, ∫(B' ⋅ D_plate ⋅ B; Γ=surface))
end
K = sum(K_plate);

In [5]:
fu = ∫(U ⋅ [0.0, 0.0, 1.0]; Γ="load")
fφ = ∫(Φ ⋅ [0.0, 1.0, 0.0]; Γ="load")
f = SystemVector([fu, fφ]);

bc_u = BoundaryCondition("clamp"; field=U, ux=0, uy=0, uz=0)
bc_φ = BoundaryCondition("clamp"; field=Φ, φx=0, φy=0, φz=0)
support = [bc_u, bc_φ];

A megoldáshoz a forgáskomponenseket csomópontonként a helyi bázisba forgatjuk. A közös él két oldalán ugyanaz a globális forgásmező szerepel. Az él csomópontjain bármelyik lemez bázisa használható, mert ott mindhárom komponens szabad. Az egyetlen lemezhez tartozó csomópontokon a normális körüli zérusenergiájú forgást elhagyjuk.

In [6]:
e1 = vectorField(Φ, [field(s; fx=a[1], fy=a[2], fz=a[3]) for (s,a) in zip(surfaces,t1)])
e2 = vectorField(Φ, [field(s; fx=a[1], fy=a[2], fz=a[3]) for (s,a) in zip(surfaces,t2)])
Qφ = rotateNodes(Φ, CoordinateSystem(e1, e2))
Q = blockdiag(spdiagm(0 => ones(ndofs(U))), Qφ.T);

φz_plate = constrainedDoFs(Φ, [BoundaryCondition(s; field=Φ, φz=0) for s in surfaces])
φz_joint = constrainedDoFs(Φ, [BoundaryCondition("joint"; field=Φ, φz=0)])
drilling = ndofs(U) .+ setdiff(φz_plate, φz_joint)

free = freeDoFs(K, support)
unrestrained_drilling = intersect(drilling, free)
@assert norm((Q' * f.a)[unrestrained_drilling]) ≤ 1e-10*norm(fφ.a) "Direct moment about a free plate normal"
T = Q[:, setdiff(free, drilling)];

In [7]:
x = T * (Symmetric(T' * K.A * T) \ (T' * f.a))
u, φ = copy(fu), copy(fφ)
u.a .= x[1:ndofs(U), :]
φ.a .= x[ndofs(U)+1:end, :];

In [8]:
showDoFResults(u; name="u", factor=20, visible=true);
# openPostProcessor()

In [9]:
openPostProcessor()

XOpenIM() failed
Fontconfig warning: using without calling FcInit()
